## 1. Situation

This notebook builds and trains a PyTorch regression model to predict a continuous target from tabular features. The task uses the diabetes regression dataset from scikit-learn, and model quality is evaluated with mean squared error (MSE).

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

X, y = load_diabetes(return_X_y=True)
X = X.astype(np.float32)
y = y.astype(np.float32)

print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Missing values in features: {np.isnan(X).sum().sum()}")
print(f"Missing values in target: {np.isnan(y).sum()}")

## 2. Data Description

The diabetes dataset contains 442 samples and 10 input features. There are no missing values in the features or target. Before training, the features are standardized so the model can optimize more effectively.

In [ ]:
# Split into train / validation / test sets
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

# Build PyTorch datasets and loaders
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f"Train samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")
print(f"Test samples: {len(test_ds)}")

## 3. Constraints

The implementation uses PyTorch, the loss function is torch.nn.MSELoss(), and the best model state is stored in a variable named best_model_state. The neural network is kept within the required layer limit by using a compact feedforward architecture with four linear layers.

## 4. Model Specification

A small feedforward neural network is defined with an input layer, three hidden layers, and an output layer. The hidden sizes are 64, 32, and 16, with ReLU activations. The architecture stays well under the 20-layer limit.

In [ ]:
class RegressionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model = RegressionNet()
print(model)
print("Linear layers:", sum(1 for m in model.modules() if isinstance(m, nn.Linear)))

## 5. Training Specification

The model is trained with Adam, a learning rate of 0.001, a batch size of 32, and up to 200 epochs. Early stopping is applied according to validation MSE, and the best model is selected based on the lowest validation MSE.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

best_val_mse = float("inf")
best_model_state = None
best_epoch = 0

train_losses = []
val_losses = []

patience = 20
epochs = 200
patience_counter = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_ds)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            preds = model(xb)
            val_loss += criterion(preds, yb).item() * xb.size(0)

    val_loss /= len(val_ds)
    val_losses.append(val_loss)

    if val_loss < best_val_mse - 1e-8:
        best_val_mse = val_loss
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch + 1
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            break

print(f"Best validation MSE: {best_val_mse:.4f} at epoch {best_epoch}")

model.load_state_dict(best_model_state)
model.eval()

test_loss = 0.0
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb)
        test_loss += criterion(preds, yb).item() * xb.size(0)

test_loss /= len(test_ds)
print(f"Test MSE: {test_loss:.4f}")

## 6. Training Curve

The training and validation MSE values are stored at each epoch and plotted as line graphs so convergence and overfitting can be visualized.